### **Installations and Imports**

In [1]:
import importlib.util
import subprocess
import sys

def is_installed(import_name):
    return importlib.util.find_spec(import_name) is not None

# package_name_on_pip : import_name_in_python
required = {
    "unsloth": "unsloth",
    "unsloth_zoo": "unsloth_zoo",
    "trl": "trl",
    "bitsandbytes": "bitsandbytes",
    "sacrebleu": "sacrebleu",
    "evaluate": "evaluate",

    # Extra safety for Gemma / tokenizer / HF training stack
    "accelerate": "accelerate",
    "transformers": "transformers",
    "datasets": "datasets",
    "peft": "peft",
    "sentencepiece": "sentencepiece",
    "protobuf": "google.protobuf",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required extra packages are already installed.")

print("Minimal installation finished.")

Missing packages: ['unsloth', 'unsloth_zoo', 'trl', 'bitsandbytes', 'sacrebleu', 'evaluate']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir unsloth unsloth_zoo trl bitsandbytes sacrebleu evaluate
Minimal installation finished.


In [1]:
# ============================================================
# Cell 1B — Verify environment
# ============================================================

import torch
import datasets
import transformers
import peft
import accelerate
import trl
import unsloth
import bitsandbytes
import sacrebleu

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available.")
    print("Do not start Qwen/Unsloth fine-tuning on CPU.")
    print("Go to Runtime → Change runtime type → GPU.")

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)
print("sacrebleu:", sacrebleu.__version__)

print("Environment check finished.")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
datasets: 4.3.0
transformers: 5.5.0
peft: 0.19.1
accelerate: 1.13.0
trl: 0.24.0
sacrebleu: 2.6.0
Environment check finished.


### **Paths and Configurations**

In [2]:
# ============================================================
# Cell 2 — Mount Google Drive and define paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")
DATA_DIR    = PROJECT_DIR / "prepared_data"
RUNS_DIR    = PROJECT_DIR / "runs"
ADAPTER_DIR = PROJECT_DIR / "final_adapters"
PRED_DIR    = PROJECT_DIR / "predictions"

for p in [PROJECT_DIR, DATA_DIR, RUNS_DIR, ADAPTER_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("PRED_DIR:", PRED_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
RUNS_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs
ADAPTER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters
PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions


In [3]:
# ============================================================
# Cell 3 — Experiment configuration
# Gemma 4 E2B + Unsloth, Alexandria MT
# Fixes: native template + explicit EOS + safer 4-bit loading on T4
# ============================================================

import torch
import random
import numpy as np

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "unsloth/gemma-4-E2B-it"
MODEL_SHORT_NAME = "gemma4_e2b_it"

SELECTED_CONFIGS_MODE = "EG_ONLY"
MANUAL_CONFIGS = ["EG"]

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# Keep your chosen experiment family: ALL LoRA r16.
LORA_MODE = "all"
LORA_R = 16

# For Gemma, start with alpha=r, not 2*r.
# This reduces update strength compared with the previous Gemma run.
LORA_ALPHA = LORA_R
LORA_DROPOUT = 0.0

GPU_TOTAL_GB = 0.0
GPU_NAME = "CPU"
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    GPU_TOTAL_GB = props.total_memory / 1024**3
    GPU_NAME = props.name

# T4 does not support BF16. Avoid full 16-bit FP16 weights on T4.
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

MODEL_LOAD_IN_4BIT = True
MODEL_LOAD_IN_16BIT = False

# If later you use A100/L4/H100 and want 16-bit/BF16 LoRA:
# MODEL_LOAD_IN_4BIT = False
# MODEL_LOAD_IN_16BIT = True

NUM_EPOCHS = 6
MAX_SEQ_LENGTH = 768

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

# Slightly lower than the first Gemma run to reduce degeneration risk.
LEARNING_RATE = 5e-5
WARMUP_RATIO = 0.03

SAVE_STEPS = 100
EVAL_STEPS = 100
LOGGING_STEPS = 10

PACKING = False
SAVE_TOTAL_LIMIT = 20

if LORA_MODE == "attn":
    LORA_GROUP_NAME = "attn_group"
elif LORA_MODE == "mlp":
    LORA_GROUP_NAME = "fnn_group"
elif LORA_MODE == "all":
    LORA_GROUP_NAME = "all_group"
else:
    raise ValueError("LORA_MODE must be 'attn', 'mlp', or 'all'.")

RUN_PROFILE = "native_template_eos_4bit_lr5e5_resume_safe_v2"


# New name = no accidental resume from the previous bad Gemma run.
EXPERIMENT_NAME = (
    f"{MODEL_SHORT_NAME}_alexandria_{SELECTED_CONFIGS_MODE.lower()}_"
    f"context{MAX_CONTEXT_TURNS}_{LORA_GROUP_NAME}_r{LORA_R}_{RUN_PROFILE}"
)

OUTPUT_DIR = RUNS_DIR / EXPERIMENT_NAME
FINAL_ADAPTER_PATH = ADAPTER_DIR / EXPERIMENT_NAME

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("GPU:", GPU_NAME)
print("GPU VRAM GB:", f"{GPU_TOTAL_GB:.2f}")
print("BF16 supported:", USE_BF16)
print("LoRA mode:", LORA_MODE)
print("LoRA r:", LORA_R)
print("LoRA alpha:", LORA_ALPHA)
print("LoRA dropout:", LORA_DROPOUT)
print("Max seq length:", MAX_SEQ_LENGTH)
print("Learning rate:", LEARNING_RATE)
print("Load in 4bit:", MODEL_LOAD_IN_4BIT)
print("Load in 16bit:", MODEL_LOAD_IN_16BIT)

Model: unsloth/gemma-4-E2B-it
Experiment: gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2
GPU: Tesla T4
GPU VRAM GB: 14.56
BF16 supported: False
LoRA mode: all
LoRA r: 16
LoRA alpha: 16
LoRA dropout: 0.0
Max seq length: 768
Learning rate: 5e-05
Load in 4bit: True
Load in 16bit: False


### **Dataset Preparation**

In [4]:
# ============================================================
# Cell 4 — List Alexandria configs and load selected configs
# ============================================================

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

DATASET_NAME = "UBC-NLP/alexandria"

available_configs = get_dataset_config_names(DATASET_NAME)
print("Available Alexandria configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"These configs are not available: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")
    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test  = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Example keys:", ds_train[0].keys())

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Available Alexandria configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Example keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [5]:
# ============================================================
# Cell 5 — Inspect one raw example
# ============================================================

sample_cfg = selected_configs[0]
sample_row = loaded[sample_cfg]["train"][0]

print("Config:", sample_cfg)
print("Keys:", sample_row.keys())

print("\nEnglish conversation:")
print(sample_row["english_conversation"])

print("\nDialectal conversation:")
print(sample_row["dialectal_conversation"])

print("\nFull row:")
sample_row

Config: EG
Keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])

English conversation:
[{'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?", 'turn_order': 1}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.", 'turn_order': 2}, {'direction': 'male -> female', 'speaker': 'Wholesale Buyer', 'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.', 'turn_order': 3}, {'direction': 'female -> male', 'speaker': 'Wholesale Seller', 'text': "Don't you worry. You will be very satisfied. My reputatio

{'conv_id': 'B7-1-0-120',
 'country': 'EG',
 'domain': 'Agriculture and farming',
 'dialect': 'Egyptian Arabic (Cairene) Dialect',
 'participants': 'Wholesale Buyer, Wholesale Seller',
 'english_conversation': [{'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': "Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?",
   'turn_order': 1},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "Good morning to you. You heard correctly. My artichokes are the best you'll find. They are top-grade, perfect for export. Let me show you a sample.",
   'turn_order': 2},
  {'direction': 'male -> female',
   'speaker': 'Wholesale Buyer',
   'text': 'Excellent. Yes, please show me. I need them to be a specific size and completely free of blemishes.',
   'turn_order': 3},
  {'direction': 'female -> male',
   'speaker': 'Wholesale Seller',
   'text': "D

#### Helper functions for robust extraction

In [6]:
# ============================================================
# Cell 6 — Helper functions for robust extraction
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def truncate_text(text, max_chars=1200):
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + " ..."

#### Flatten Alexandria conversations into SFT examples and Save

In [7]:
# ============================================================
# Cell 7 — Flatten Alexandria conversations
# ============================================================

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [8]:
# ============================================================
# Cell 8 — Save prepared flattened data
# ============================================================

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl  = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved train:", train_jsonl)
print("Saved eval:", eval_jsonl)

Saved train: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
Saved eval: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


### **Build prompt and chat messages**

In [9]:
# ============================================================
# Cell 9 — Build prompt and chat messages
# ============================================================

from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")
        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use natural local dialectal Arabic, not Modern Standard Arabic unless it is natural in context.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"]  = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

Example messages:


[{'role': 'system',
  'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Return only the translation, without explanation.'},
 {'role': 'user',
  'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nMetadata:\nCountry/config: EG\nTarget dialect: Egyptian Arabic (Cairene) Dialect\nDomain: Agriculture and farming\nCurrent speaker: Wholesale Buyer\nSpeaker-to-addressee gender direction: male -> female\n\nPrevious English dialogue context:\nNo previous context.\n\nCurrent English turn:\nGood morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Obour comes from your section, is that right?\n\nRules:\n- Preserve the meaning exactly.\n- Use natural local dialectal Arabic, not Modern Standard Arabic unless it is natural in context.\n- Preserve names, numbers, named entities, and technical terms when appr

### **Load Gemma 4 E2B with Unsloth**

In [10]:
# ============================================================
# Cell 10 — Load Gemma 4 E2B with Unsloth
# Fixes: avoid full FP16 weights on T4; preserve tokenizer/EOS details
# ============================================================

import torch

try:
    from unsloth import FastLanguageModel
    UnslothModel = FastLanguageModel
    print("Using unsloth.FastLanguageModel")
except Exception as e:
    print("FastLanguageModel import failed:", repr(e))
    from unsloth import FastModel
    UnslothModel = FastModel
    print("Using unsloth.FastModel fallback")

# For 4-bit loading, dtype=None lets Unsloth choose safely.
# On T4, compute may still be FP16, but model weights are not full FP16.
dtype = None if MODEL_LOAD_IN_4BIT else (torch.bfloat16 if USE_BF16 else torch.float16)

def load_base_model():
    kwargs = dict(
        model_name = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
    )

    try:
        return UnslothModel.from_pretrained(
            **kwargs,
            dtype = dtype,
            load_in_4bit = MODEL_LOAD_IN_4BIT,
            load_in_16bit = MODEL_LOAD_IN_16BIT,
        )
    except TypeError as e:
        print("Loader did not accept load_in_16bit. Retrying older signature.")
        print("Reason:", repr(e))
        return UnslothModel.from_pretrained(
            **kwargs,
            dtype = dtype,
            load_in_4bit = MODEL_LOAD_IN_4BIT,
        )

model, tokenizer = load_base_model()

if hasattr(tokenizer, "tokenizer"):
    print("Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.")
    tokenizer = tokenizer.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

STOP_TOKEN_IDS = []
if tokenizer.eos_token_id is not None:
    STOP_TOKEN_IDS.append(int(tokenizer.eos_token_id))

for tok in ["<end_of_turn>", "<|im_end|>"]:
    try:
        tok_id = tokenizer.convert_tokens_to_ids(tok)
        if tok_id is not None and tok_id != tokenizer.unk_token_id and tok_id not in STOP_TOKEN_IDS:
            STOP_TOKEN_IDS.append(int(tok_id))
    except Exception:
        pass

if len(STOP_TOKEN_IDS) == 0:
    raise RuntimeError("Could not identify any EOS/stop token id.")

print("Loaded:", MODEL_NAME)
print("dtype passed to loader:", dtype)
print("Tokenizer type:", type(tokenizer))
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("stop token ids:", STOP_TOKEN_IDS)
print("chat template exists:", tokenizer.chat_template is not None)

if tokenizer.chat_template is not None:
    print("\nChat template preview:")
    print(str(tokenizer.chat_template)[:1000])

Using unsloth.FastLanguageModel
==((====))==  Unsloth 2026.5.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/301k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/16.8k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Tokenizer object has internal tokenizer. Using tokenizer.tokenizer.
Loaded: unsloth/gemma-4-E2B-it
dtype passed to loader: None
Tokenizer type: <class 'transformers.models.gemma.tokenization_gemma.GemmaTokenizer'>
pad token: <pad> 0
eos token: <eos> 1
stop token ids: [1]
chat template exists: True

Chat template preview:
{%- macro format_parameters(properties, required, filter_keys=false) -%}
    {%- set standard_keys = ['description', 'type', 'properties', 'required', 'nullable'] -%}
    {%- set ns = namespace(found_first=false) -%}
    {%- for key, value in properties | dictsort -%}
        {%- set add_comma = false -%}
        {%- if not filter_keys or key not in standard_keys -%}
            {%- if ns.found_first %},{% endif -%}
            {%- set ns.found_first = true -%}
            {{ key }}:{
            {%- if value['description'] -%}
                description:<|"|>{{ value['description'] }}<|"|>
                {%- set add_comma = true -%}
            {%- endif -%}
       

### **Apply Chat Template**

In [11]:
# ============================================================
# Cell 11 — Apply Gemma native chat template
# Fix: robust Gemma turn markers for response-only masking
# ============================================================

def messages_to_gemma_chat(messages, include_assistant=True):
    """
    Gemma 4 uses standard chat roles.
    To avoid system-role incompatibility in some tokenizers, we fold system into user.
    """
    system_text = ""
    user_text = ""
    assistant_text = ""

    for m in messages:
        role = m.get("role", "")
        content = str(m.get("content", ""))

        if role == "system":
            system_text = content.strip()
        elif role == "user":
            user_text = content.strip()
        elif role in ["assistant", "model"]:
            assistant_text = content.strip()

    folded_user = (
        "System instruction:\n"
        f"{system_text}\n\n"
        "User request:\n"
        f"{user_text}"
    ).strip()

    out = [{"role": "user", "content": folded_user}]

    if include_assistant:
        out.append({"role": "assistant", "content": assistant_text})

    return out


def apply_chat_template_robust(messages, tokenize=False, add_generation_prompt=False):
    """
    First try assistant role. If tokenizer expects model role, retry assistant -> model.
    """
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=tokenize,
            add_generation_prompt=add_generation_prompt,
        )
    except Exception as e1:
        converted = []
        for m in messages:
            mm = dict(m)
            if mm.get("role") == "assistant":
                mm["role"] = "model"
            converted.append(mm)

        try:
            return tokenizer.apply_chat_template(
                converted,
                tokenize=tokenize,
                add_generation_prompt=add_generation_prompt,
            )
        except Exception as e2:
            print("apply_chat_template failed with assistant role:", repr(e1))
            print("apply_chat_template failed with model role:", repr(e2))
            raise


def apply_gemma_template(example):
    chat_messages = messages_to_gemma_chat(
        example["messages"],
        include_assistant=True,
    )

    text = apply_chat_template_robust(
        chat_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Safety: make sure each training example ends with EOS.
    if tokenizer.eos_token is not None and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token

    return {"text": text}


remove_cols_train = [c for c in train_dataset.column_names if c == "messages"]
remove_cols_eval  = [c for c in eval_dataset.column_names if c == "messages"]

train_dataset_text = train_dataset.map(
    apply_gemma_template,
    remove_columns=remove_cols_train,
)

eval_dataset_text = eval_dataset.map(
    apply_gemma_template,
    remove_columns=remove_cols_eval,
)

print(train_dataset_text)
print(eval_dataset_text)

print("\nFormatted Gemma example:")
print(train_dataset_text[0]["text"][:3000])


# ------------------------------------------------------------
# Correct Gemma markers for train_on_responses_only
# ------------------------------------------------------------

dummy_messages = [
    {"role": "user", "content": "Translate this sentence."},
    {"role": "assistant", "content": "الترجمة التجريبية"},
]

dummy_rendered = apply_chat_template_robust(
    dummy_messages,
    tokenize=False,
    add_generation_prompt=False,
)

print("\nDummy rendered template:")
print(dummy_rendered)

# Your actual tokenizer rendered this style:
# <bos><|turn>user
# ...
# <turn|>
# <|turn>model
# ...
# <turn|>
if "<|turn>user\n" in dummy_rendered and "<|turn>model\n" in dummy_rendered:
    INSTRUCTION_PART = "<|turn>user\n"
    RESPONSE_PART = "<|turn>model\n"

elif "<|turn>user\n" in dummy_rendered and "<|turn>assistant\n" in dummy_rendered:
    INSTRUCTION_PART = "<|turn>user\n"
    RESPONSE_PART = "<|turn>assistant\n"

elif "<start_of_turn>user\n" in dummy_rendered and "<start_of_turn>model\n" in dummy_rendered:
    INSTRUCTION_PART = "<start_of_turn>user\n"
    RESPONSE_PART = "<start_of_turn>model\n"

elif "<start_of_turn>user\n" in dummy_rendered and "<start_of_turn>assistant\n" in dummy_rendered:
    INSTRUCTION_PART = "<start_of_turn>user\n"
    RESPONSE_PART = "<start_of_turn>assistant\n"

else:
    raise RuntimeError(
        "Could not infer Gemma user/model markers from chat template. "
        "Inspect dummy_rendered above."
    )

print("\nInstruction marker used for masking:")
print(repr(INSTRUCTION_PART))

print("Response marker used for masking:")
print(repr(RESPONSE_PART))


# ------------------------------------------------------------
# Pre-mask sanity check
# ------------------------------------------------------------

example_text = train_dataset_text[0]["text"]

if INSTRUCTION_PART not in example_text:
    raise RuntimeError(
        f"INSTRUCTION_PART not found in formatted training example: {repr(INSTRUCTION_PART)}"
    )

if RESPONSE_PART not in example_text:
    raise RuntimeError(
        f"RESPONSE_PART not found in formatted training example: {repr(RESPONSE_PART)}"
    )

after_response_marker = example_text.split(RESPONSE_PART, 1)[1]

print("\nText immediately after RESPONSE_PART:")
print(after_response_marker[:500])

print("\nMarker sanity check passed.")

Map:   0%|          | 0/3108 [00:00<?, ? examples/s]

Map:   0%|          | 0/1118 [00:00<?, ? examples/s]

Dataset({
    features: ['source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic', 'text'],
    num_rows: 3108
})
Dataset({
    features: ['source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic', 'text'],
    num_rows: 1118
})

Formatted Gemma example:
<bos><|turn>user
System instruction:
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Return only the translation, without explanation.

User request:
Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Metadata:
Country/config: EG
Target dialect: Egyptian Arabic (Cairene) Dialect
Domain: Agriculture and farming
Current speaker: Wholesale Buyer
Speaker-to-addressee gender direction: male -> female

Previous English dialogue context:
No previous context.

Current English turn:
Good morning. I'm looking to source 10 tons of premium artichokes for export. People say the best quality in Ob

#### **Configure LoRA target modules**

In [12]:
# ============================================================
# Cell 12 — Configure LoRA
# Gemma 4 E2B ALL r16: attention + MLP
# Fixes: gentler alpha and no dropout for Gemma baseline
# ============================================================

if LORA_MODE == "attn":
    requested_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

elif LORA_MODE == "mlp":
    requested_modules = ["gate_proj", "up_proj", "down_proj"]

elif LORA_MODE == "all":
    requested_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]

else:
    raise ValueError("LORA_MODE must be 'attn', 'mlp', or 'all'.")

module_names = [name for name, _ in model.named_modules()]
available_leaf_names = sorted(set(name.split(".")[-1] for name in module_names))

TARGET_MODULES = [m for m in requested_modules if m in available_leaf_names]
missing_modules = [m for m in requested_modules if m not in available_leaf_names]

print("Requested target modules:", requested_modules)
print("Resolved target modules:", TARGET_MODULES)

if missing_modules:
    print("Missing requested modules:", missing_modules)
    print("Available module suffix sample:", available_leaf_names[:100])

if not TARGET_MODULES:
    raise RuntimeError(
        "No LoRA target modules were resolved. "
        "Inspect model.named_modules() and update target_modules."
    )

model = UnslothModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()

# Optional sanity check: ensure we are not training audio/vision towers.
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
bad = [
    n for n in trainable
    if any(k in n.lower() for k in ["audio", "vision", "image", "tower", "projector"])
]
print("Trainable tensors:", len(trainable))
print("Trainable multimodal/audio/vision tensors:", len(bad))
for n in bad[:30]:
    print("WARNING trainable non-text tensor:", n)

Requested target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
Resolved target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


trainable params: 31,039,488 || all params: 5,154,217,504 || trainable%: 0.6022
Trainable tensors: 786
Trainable multimodal/audio/vision tensors: 296
WARNING trainable non-text tensor: base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.linear.lora_A.default.weight
WARNING trainable non-text tensor: base_model.model.model.vision_tower.encoder.layers.0.self_attn.q_proj.linear.lora_B.default.weight
WARNING trainable non-text tensor: base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.linear.lora_A.default.weight
WARNING trainable non-text tensor: base_model.model.model.vision_tower.encoder.layers.0.self_attn.k_proj.linear.lora_B.default.weight
WARNING trainable non-text tensor: base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.linear.lora_A.default.weight
WARNING trainable non-text tensor: base_model.model.model.vision_tower.encoder.layers.0.self_attn.v_proj.linear.lora_B.default.weight
WARNING trainable non-text tensor: base_model.

#### Check for existing checkpoints

In [13]:
# ============================================================
# Cell 13 — Check for existing checkpoints safely
# Supports resume, but blocks incompatible old folders
# ============================================================

from transformers.trainer_utils import get_last_checkpoint
from pathlib import Path
import json

FORCE_RESTART = False

RUN_SIGNATURE_PATH = OUTPUT_DIR / "run_signature.json"

CURRENT_RUN_SIGNATURE = {
    "model_name": MODEL_NAME,
    "model_short_name": MODEL_SHORT_NAME,
    "selected_configs_mode": SELECTED_CONFIGS_MODE,
    "manual_configs": MANUAL_CONFIGS,
    "max_context_turns": MAX_CONTEXT_TURNS,
    "use_previous_english_context": USE_PREVIOUS_ENGLISH_CONTEXT,
    "use_metadata": USE_METADATA,
    "template_version": "gemma_native_chat_template_v1",
    "lora_mode": LORA_MODE,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "num_epochs": NUM_EPOCHS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "model_load_in_4bit": MODEL_LOAD_IN_4BIT,
    "model_load_in_16bit": MODEL_LOAD_IN_16BIT,
    "run_profile": RUN_PROFILE,
    "experiment_name": EXPERIMENT_NAME,
}

last_checkpoint = None

if OUTPUT_DIR.exists() and RUN_SIGNATURE_PATH.exists():
    old_signature = json.loads(RUN_SIGNATURE_PATH.read_text(encoding="utf-8"))

    if old_signature != CURRENT_RUN_SIGNATURE:
        print("Existing run signature does not match current config.")
        print("\nExisting signature:")
        print(json.dumps(old_signature, indent=2, ensure_ascii=False))
        print("\nCurrent signature:")
        print(json.dumps(CURRENT_RUN_SIGNATURE, indent=2, ensure_ascii=False))

        raise RuntimeError(
            "Refusing to resume from an incompatible experiment folder. "
            "Use a new RUN_PROFILE / EXPERIMENT_NAME, or intentionally delete/rename OUTPUT_DIR."
        )

elif OUTPUT_DIR.exists() and not RUN_SIGNATURE_PATH.exists():
    existing_ckpts = list(OUTPUT_DIR.glob("checkpoint-*"))

    if existing_ckpts and not FORCE_RESTART:
        raise RuntimeError(
            f"Found checkpoints in {OUTPUT_DIR}, but no run_signature.json exists. "
            "This may be an old/incompatible run. "
            "Use a new RUN_PROFILE/EXPERIMENT_NAME or set FORCE_RESTART=True intentionally."
        )

if FORCE_RESTART:
    print("FORCE_RESTART=True.")
    print("Training will start from scratch. Existing checkpoints are ignored.")
    last_checkpoint = None

else:
    RUN_SIGNATURE_PATH.write_text(
        json.dumps(CURRENT_RUN_SIGNATURE, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))

    if last_checkpoint:
        print("Found compatible checkpoint. Training will resume from:")
        print(last_checkpoint)
    else:
        print("No compatible checkpoint found. Training will start from scratch.")

Found compatible checkpoint. Training will resume from:
/content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-900


### **Build SFTTrainer**

In [14]:
# ============================================================
# Cell 14 — Build SFTTrainer
# Resume-safe checkpoint settings
# ============================================================

from trl import SFTTrainer, SFTConfig
import torch

use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
use_fp16 = bool(torch.cuda.is_available() and not torch.cuda.is_bf16_supported())

sft_args = SFTConfig(
    output_dir = str(OUTPUT_DIR),

    num_train_epochs = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size = 1,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,

    learning_rate = LEARNING_RATE,
    warmup_ratio = WARMUP_RATIO,
    lr_scheduler_type = "cosine",

    optim = "adamw_8bit",
    weight_decay = 0.01,

    logging_steps = LOGGING_STEPS,

    eval_strategy = "steps",
    eval_steps = EVAL_STEPS,

    save_strategy = "steps",
    save_steps = SAVE_STEPS,
    save_total_limit = SAVE_TOTAL_LIMIT,

    # We manually load the true best checkpoint in Cell 17.
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,

    fp16 = use_fp16,
    bf16 = use_bf16,

    seed = SEED,
    dataset_num_proc = 2,
    report_to = "none",

    packing = PACKING,
    dataset_text_field = "text",
    max_length = MAX_SEQ_LENGTH,
)

try:
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset_text,
        eval_dataset = eval_dataset_text,
        args = sft_args,
    )
except TypeError:
    trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = train_dataset_text,
        eval_dataset = eval_dataset_text,
        args = sft_args,
    )

print("Trainer ready.")
print("Output dir:", OUTPUT_DIR)
print("Save steps:", SAVE_STEPS)
print("Eval steps:", EVAL_STEPS)
print("Save total limit:", SAVE_TOTAL_LIMIT)
print("fp16:", use_fp16)
print("bf16:", use_bf16)
print("Resume checkpoint:", last_checkpoint)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3108 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1118 [00:00<?, ? examples/s]

Trainer ready.
Output dir: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2
Save steps: 100
Eval steps: 100
Save total limit: 20
fp16: True
bf16: False
Resume checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-900


#### Response-only training with manual markers

In [15]:
# ============================================================
# Cell 15 — Train on assistant/model response only
# Fix: verify dataset is not emptied after masking
# ============================================================

try:
    from unsloth.chat_templates import train_on_responses_only

    print("Before response-only masking:")
    print("Train size:", len(trainer.train_dataset))
    print("Eval size:", len(trainer.eval_dataset))

    trainer = train_on_responses_only(
        trainer,
        instruction_part = INSTRUCTION_PART,
        response_part = RESPONSE_PART,
    )

    print("\nEnabled response-only training.")
    print("Instruction marker:", repr(INSTRUCTION_PART))
    print("Response marker:", repr(RESPONSE_PART))

    print("\nAfter response-only masking:")
    print("Train size:", len(trainer.train_dataset))
    print("Eval size:", len(trainer.eval_dataset))

except Exception as e:
    raise RuntimeError(
        "Could not enable response-only training. "
        "Do not continue this Gemma run until masking is fixed."
    ) from e


# ------------------------------------------------------------
# Critical safety check
# ------------------------------------------------------------

if len(trainer.train_dataset) == 0:
    raise RuntimeError(
        "train_on_responses_only removed all training examples. "
        "This means RESPONSE_PART / INSTRUCTION_PART still do not match the formatted text."
    )

if len(trainer.eval_dataset) == 0:
    raise RuntimeError(
        "train_on_responses_only removed all eval examples. "
        "This means RESPONSE_PART / INSTRUCTION_PART still do not match the formatted text."
    )


# ------------------------------------------------------------
# Diagnostic: verify only Arabic answer tokens are supervised
# ------------------------------------------------------------

sample = trainer.train_dataset[0]

input_ids = sample["input_ids"]
labels = sample["labels"]

visible_label_ids = [
    int(tok_id)
    for tok_id, lab in zip(input_ids, labels)
    if int(lab) != -100
]

print("\nNumber of supervised tokens:", len(visible_label_ids))

if len(visible_label_ids) == 0:
    raise RuntimeError("No supervised tokens found. Response-only masking failed.")

decoded_supervised = tokenizer.decode(
    visible_label_ids,
    skip_special_tokens=False,
)

print("\nDecoded supervised part:")
print(decoded_supervised[:1500])

bad_fragments = [
    "System instruction:",
    "User request:",
    "Translate the current English",
    "Metadata:",
    "Current English:",
    "Previous context:",
]

if any(fragment in decoded_supervised for fragment in bad_fragments):
    raise RuntimeError(
        "Masking looks wrong: prompt text is supervised. "
        "Fix INSTRUCTION_PART / RESPONSE_PART before training."
    )

print("\nMasking diagnostic passed: only the model/Arabic answer span is supervised.")

Before response-only masking:
Train size: 3108
Eval size: 1118


Map (num_proc=4):   0%|          | 0/3108 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/3108 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1118 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/1118 [00:00<?, ? examples/s]


Enabled response-only training.
Instruction marker: '<|turn>user\n'
Response marker: '<|turn>model\n'

After response-only masking:
Train size: 3108
Eval size: 1118

Number of supervised tokens: 40

Decoded supervised part:
صباح الخير، عايز عشرة طن من الخرشوف الكويس للتصدير، بيقولوا ان احسن جودة في سوق العبور بتيجي من عندكم، صحيح؟<turn|>
<eos>

Masking diagnostic passed: only the model/Arabic answer span is supervised.


### **Training**

In [16]:
# ============================================================
# Cell 16 — Train or resume
# Resume from latest compatible checkpoint if runtime disconnects
# ============================================================

if last_checkpoint:
    print("Resuming training from latest compatible checkpoint:")
    print(last_checkpoint)

    trainer_stats = trainer.train(
        resume_from_checkpoint = last_checkpoint
    )

else:
    print("Starting training from scratch.")
    trainer_stats = trainer.train()

print("Training finished.")
print(trainer_stats)

trainer.save_state()

print("Trainer state saved to:")
print(OUTPUT_DIR)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Resuming training from latest compatible checkpoint:
/content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-900


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,108 | Num Epochs = 6 | Total steps = 2,334
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss,Validation Loss
1000,0.175723,3.073067
1100,0.167605,3.074597


Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1100/tokenizer_config.json.


Step,Training Loss,Validation Loss
1000,0.175723,3.073067
1100,0.167605,3.074597
1200,0.131972,3.124843
1300,0.128402,3.167445
1400,0.128469,3.151436


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1400/tokenizer_config.json.


KeyboardInterrupt: 

### **Save final LoRA adapter**

In [17]:
# ============================================================
# Cell 17 — Find, load, and save BEST checkpoint
# Fixes: fallback loader respects 4-bit/16-bit config
# ============================================================

from pathlib import Path
import json
import torch

def find_best_checkpoint_from_logs(output_dir):
    output_dir = Path(output_dir)

    state_files = list(output_dir.rglob("trainer_state.json"))
    if not state_files:
        raise FileNotFoundError(f"No trainer_state.json found under: {output_dir}")

    best_state_file = None
    best_state = None
    max_global_step = -1

    for sf in state_files:
        try:
            state = json.loads(sf.read_text())
            global_step = int(state.get("global_step", -1))

            if global_step > max_global_step:
                max_global_step = global_step
                best_state_file = sf
                best_state = state
        except Exception:
            continue

    if best_state is None:
        raise RuntimeError("Could not read any valid trainer_state.json")

    official_best = best_state.get("best_model_checkpoint", None)
    official_metric = best_state.get("best_metric", None)

    if official_best is not None:
        official_best_path = Path(official_best)

        if not official_best_path.exists():
            official_best_path = output_dir / official_best_path.name

        if official_best_path.exists():
            best_step = int(official_best_path.name.replace("checkpoint-", ""))
            return official_best_path, best_step, official_metric, None, best_state_file

    eval_rows = []

    for item in best_state.get("log_history", []):
        if "eval_loss" in item and "step" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_loss"]),
                "epoch": item.get("epoch", None),
            })

    if not eval_rows:
        raise RuntimeError("No eval_loss records found in trainer_state.json")

    best_row = min(eval_rows, key=lambda x: x["eval_loss"])

    best_step = best_row["step"]
    best_eval_loss = best_row["eval_loss"]
    best_epoch = best_row["epoch"]

    best_checkpoint_path = output_dir / f"checkpoint-{best_step}"

    if not best_checkpoint_path.exists():
        existing = sorted([p.name for p in output_dir.glob("checkpoint-*")])
        raise FileNotFoundError(
            f"Best checkpoint is missing: {best_checkpoint_path}\n"
            f"Best step from logs = {best_step}, eval_loss = {best_eval_loss}\n"
            f"Existing checkpoints: {existing}\n"
            f"Rerun with SAVE_TOTAL_LIMIT high enough."
        )

    return best_checkpoint_path, best_step, best_eval_loss, best_epoch, best_state_file


BEST_CHECKPOINT_PATH, BEST_STEP, BEST_EVAL_LOSS, BEST_EPOCH, BEST_STATE_FILE = find_best_checkpoint_from_logs(OUTPUT_DIR)

print("Best checkpoint:")
print("  path:", BEST_CHECKPOINT_PATH)
print("  step:", BEST_STEP)
print("  epoch:", BEST_EPOCH)
print("  eval_loss:", BEST_EVAL_LOSS)
print("  trainer_state:", BEST_STATE_FILE)

print("\nLoading best checkpoint into model...")

try:
    trainer._load_from_checkpoint(str(BEST_CHECKPOINT_PATH), model=trainer.model)
    model = trainer.model
    print("Loaded best checkpoint using trainer._load_from_checkpoint().")

except Exception as e:
    print("trainer._load_from_checkpoint failed:")
    print(repr(e))
    print("Trying PEFT fallback load...")

    from peft import PeftModel

    try:
        base_model, tokenizer = UnslothModel.from_pretrained(
            model_name = MODEL_NAME,
            max_seq_length = MAX_SEQ_LENGTH,
            dtype = dtype,
            load_in_4bit = MODEL_LOAD_IN_4BIT,
            load_in_16bit = MODEL_LOAD_IN_16BIT,
        )
    except TypeError:
        base_model, tokenizer = UnslothModel.from_pretrained(
            model_name = MODEL_NAME,
            max_seq_length = MAX_SEQ_LENGTH,
            dtype = dtype,
            load_in_4bit = MODEL_LOAD_IN_4BIT,
        )

    if hasattr(tokenizer, "tokenizer"):
        tokenizer = tokenizer.tokenizer

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    model = PeftModel.from_pretrained(base_model, str(BEST_CHECKPOINT_PATH))
    trainer.model = model

    print("Loaded best checkpoint using PEFT fallback.")

BEST_ADAPTER_PATH = ADAPTER_DIR / f"{EXPERIMENT_NAME}_best_step{BEST_STEP}"
BEST_ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(BEST_ADAPTER_PATH))
tokenizer.save_pretrained(str(BEST_ADAPTER_PATH))

print("\nSaved BEST adapter to:")
print(BEST_ADAPTER_PATH)

print("\nActive model for inference is now the BEST checkpoint.")
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("BEST_STEP:", BEST_STEP)
print("BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

Best checkpoint:
  path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1000
  step: 1000
  epoch: None
  eval_loss: 3.0730669498443604
  trainer_state: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1400/trainer_state.json

Loading best checkpoint into model...
Loaded best checkpoint using trainer._load_from_checkpoint().


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2_best_step1000/tokenizer_config.json.



Saved BEST adapter to:
/content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2_best_step1000

Active model for inference is now the BEST checkpoint.
BEST_CHECKPOINT_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1000
BEST_STEP: 1000
BEST_EVAL_LOSS: 3.0730669498443604


### **Quick Inference**

In [18]:
# ============================================================
# Cell 18 — Quick inference function
# Fixes: native Gemma chat prompt + decode only new tokens + explicit EOS
# ============================================================

import torch

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError(
        "BEST_CHECKPOINT_PATH is not defined. "
        "Run Cell 17 first to load the best checkpoint before inference."
    )

print("Inference will use BEST checkpoint:")
print("BEST_CHECKPOINT_PATH:", BEST_CHECKPOINT_PATH)
print("BEST_STEP:", BEST_STEP)
print("BEST_EVAL_LOSS:", BEST_EVAL_LOSS)

try:
    UnslothModel.for_inference(model)
except Exception as e:
    print("for_inference not available or not needed:", repr(e))

def build_inference_messages(row):
    train_like_messages = row_to_messages({
        **row,
        "target_arabic": "",
    })

    return messages_to_gemma_chat(train_like_messages, include_assistant=False)

def clean_generation_text(text):
    text = str(text)

    for tok in [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<end_of_turn>",
        "<start_of_turn>",
        "<|endoftext|>",
        "<|im_end|>",
    ]:
        if tok:
            text = text.replace(tok, "")

    for marker in ["assistant", "model"]:
        if text.strip().startswith(marker):
            text = text.strip()[len(marker):].strip()

    return text.strip()

def generate_translation_from_row(row, max_new_tokens=96):
    messages = build_inference_messages(row)

    prompt = apply_chat_template_robust(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    eos_arg = STOP_TOKEN_IDS if len(STOP_TOKEN_IDS) > 1 else STOP_TOKEN_IDS[0]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            repetition_penalty=1.10,
            no_repeat_ngram_size=4,
            eos_token_id=eos_arg,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = outputs[0][input_len:]
    raw_answer = tokenizer.decode(generated_ids, skip_special_tokens=False)
    answer = clean_generation_text(raw_answer)

    full_decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

    return answer, full_decoded

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

pred, raw = generate_translation_from_row(sample)

print("Country/config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])
print("\nEnglish:")
print(sample["source_text"])
print("\nReference Arabic:")
print(sample["target_arabic"])
print("\nPrediction:")
print(pred)
print("\nRaw decoded debug:")
print(raw[-2000:])

Inference will use BEST checkpoint:
BEST_CHECKPOINT_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1000
BEST_STEP: 1000
BEST_EVAL_LOSS: 3.0730669498443604
Country/config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Construction and real estate

English:
Engineer, good morning. Before you run your cables on the third floor, let's coordinate the wall chases.

Reference Arabic:
صباح الخير يا هندسه. قبل ما تمد الكابلات في الدور التالت، خلينا نتفق على مجاري الحيطان.

Prediction:
يا باشمهندس، صباح الخير. قبل ما تشغل الكابلات في الدور التالت، خلينا نتفق على مجاري الحيطة.<turn|><turn|><turn|><turn|>

Raw decoded debug:
<bos><|turn>user
System instruction:
You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Return only the translation, without explanation.

User request:
Task:
Translate the c

### Generate predictions for a small eval sample

In [ ]:
# ============================================================
# Cell 19 — Generate predictions on FULL eval/test set
# Uses BEST checkpoint loaded in Cell 17
# Supports safe resume if runtime disconnects during prediction
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import time
import os

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first. The best checkpoint is not loaded.")

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

if "BEST_EVAL_LOSS" not in globals():
    raise RuntimeError("BEST_EVAL_LOSS is not defined. Run Cell 17 first.")

# None = full eval/test set
EVAL_LIMIT = None

SAVE_EVERY = 25
STORE_RAW_OUTPUT = False

# IMPORTANT:
# False = resume partial prediction file if runtime disconnects.
# True  = delete existing prediction file and regenerate from scratch.
FORCE_REGENERATE_PREDICTIONS = False

eval_tag = f"best_step{BEST_STEP}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"
tmp_pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.partial.csv"

print("Experiment:", EXPERIMENT_NAME)
print("Using BEST checkpoint:", BEST_CHECKPOINT_PATH)
print("Best step:", BEST_STEP)
print("Best eval_loss:", BEST_EVAL_LOSS)
print("Saving predictions to:", pred_path)
print("Temporary partial file:", tmp_pred_path)

full_eval_df = eval_df.reset_index(drop=True).copy()

if EVAL_LIMIT is not None:
    full_eval_df = full_eval_df.iloc[:EVAL_LIMIT].copy()

expected_n = len(full_eval_df)
expected_ids = set(full_eval_df["source_id"].astype(str).tolist())

print("Total eval/test examples to evaluate:", expected_n)

# ------------------------------------------------------------
# Decide whether to resume existing prediction file
# ------------------------------------------------------------

resume_path = None

if FORCE_REGENERATE_PREDICTIONS:
    print("FORCE_REGENERATE_PREDICTIONS=True")
    print("Existing prediction files will be ignored and overwritten.")
    pred_rows = []
    done_ids = set()

else:
    if tmp_pred_path.exists():
        resume_path = tmp_pred_path
    elif pred_path.exists():
        resume_path = pred_path

    if resume_path is not None:
        print("Found existing prediction file:")
        print(resume_path)

        existing_df = pd.read_csv(resume_path)

        required_cols = {
            "source_id",
            "prediction",
            "best_step",
            "model_checkpoint",
        }

        missing_cols = required_cols - set(existing_df.columns)

        if missing_cols:
            print("Existing prediction file is incompatible.")
            print("Missing columns:", missing_cols)
            print("Starting prediction from scratch.")
            pred_rows = []
            done_ids = set()

        else:
            existing_df["source_id"] = existing_df["source_id"].astype(str)

            # Keep only rows belonging to the current eval set.
            existing_df = existing_df[existing_df["source_id"].isin(expected_ids)].copy()

            # Keep only rows generated from the same best step.
            existing_df = existing_df[existing_df["best_step"].astype(str) == str(BEST_STEP)].copy()

            # Remove duplicate source_ids, keeping the first completed row.
            existing_df = existing_df.drop_duplicates(subset=["source_id"], keep="first").copy()

            pred_rows = existing_df.to_dict("records")
            done_ids = set(existing_df["source_id"].astype(str).tolist())

            print(f"Resuming prediction generation.")
            print(f"Already completed examples for this best_step: {len(done_ids)}")

    else:
        print("No existing prediction file found. Starting from scratch.")
        pred_rows = []
        done_ids = set()

# ------------------------------------------------------------
# If already complete, skip generation
# ------------------------------------------------------------

if len(done_ids) == expected_n:
    print("Prediction file already contains all expected eval examples.")
    pred_df = pd.DataFrame(pred_rows)

    # Save normalized final copy.
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    if tmp_pred_path.exists() and tmp_pred_path != pred_path:
        try:
            tmp_pred_path.unlink()
            print("Removed partial file after confirming full completion.")
        except Exception as e:
            print("Could not remove partial file:", repr(e))

    display(pred_df.head())

else:
    print(f"Remaining examples to generate: {expected_n - len(done_ids)}")

    start_time = time.time()

    for _, row in tqdm(full_eval_df.iterrows(), total=len(full_eval_df)):
        row_dict = row.to_dict()
        source_id = str(row_dict["source_id"])

        if source_id in done_ids:
            continue

        try:
            pred, raw = generate_translation_from_row(row_dict)

            out_row = {
                "source_id": row_dict["source_id"],
                "config": row_dict.get("config", ""),
                "dialect": row_dict.get("dialect", ""),
                "domain": row_dict.get("domain", ""),
                "source_text": row_dict["source_text"],
                "reference_arabic": row_dict["target_arabic"],
                "prediction": pred,
                "model_name": MODEL_NAME,
                "experiment_name": EXPERIMENT_NAME,
                "model_checkpoint": str(BEST_CHECKPOINT_PATH),
                "best_step": BEST_STEP,
                "best_eval_loss": BEST_EVAL_LOSS,
            }

            if STORE_RAW_OUTPUT:
                out_row["raw_output"] = raw

        except Exception as e:
            out_row = {
                "source_id": row_dict.get("source_id", ""),
                "config": row_dict.get("config", ""),
                "dialect": row_dict.get("dialect", ""),
                "domain": row_dict.get("domain", ""),
                "source_text": row_dict.get("source_text", ""),
                "reference_arabic": row_dict.get("target_arabic", ""),
                "prediction": "",
                "generation_error": repr(e),
                "model_name": MODEL_NAME,
                "experiment_name": EXPERIMENT_NAME,
                "model_checkpoint": str(BEST_CHECKPOINT_PATH),
                "best_step": BEST_STEP,
                "best_eval_loss": BEST_EVAL_LOSS,
            }

        pred_rows.append(out_row)
        done_ids.add(source_id)

        if len(pred_rows) % SAVE_EVERY == 0:
            tmp_df = pd.DataFrame(pred_rows)
            tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
            tmp_df.to_csv(tmp_pred_path, index=False, encoding="utf-8-sig")
            print(f"Saved partial predictions: {len(tmp_df)} rows")

    pred_df = pd.DataFrame(pred_rows)
    pred_df = pred_df.drop_duplicates(subset=["source_id"], keep="first")

    # Save final prediction file.
    pred_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    # Keep or remove partial file after successful final save.
    if tmp_pred_path.exists():
        try:
            tmp_pred_path.unlink()
            print("Removed partial prediction file after successful final save.")
        except Exception as e:
            print("Could not remove partial file:", repr(e))

    elapsed = time.time() - start_time

    actual_ids = set(pred_df["source_id"].astype(str).tolist())

    missing_ids = expected_ids - actual_ids
    extra_ids = actual_ids - expected_ids

    print("\nDone.")
    print("Saved predictions to:", pred_path)
    print("Total rows saved:", len(pred_df))
    print("Expected eval/test rows:", expected_n)
    print(f"Elapsed time: {elapsed / 60:.2f} minutes")

    if missing_ids:
        raise RuntimeError(
            f"Prediction file is incomplete. Missing {len(missing_ids)} eval examples."
        )

    if extra_ids:
        print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

    if len(pred_df) == expected_n:
        print("Full eval/test set was evaluated successfully.")
    else:
        print("Warning: row count differs from expected eval size.")

    display(pred_df.head())

Experiment: gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2
Using BEST checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2/checkpoint-1000
Best step: 1000
Best eval_loss: 3.0730669498443604
Saving predictions to: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2_best_step1000.csv
Temporary partial file: /content/drive/MyDrive/alexandria_qwen35_sft/predictions/full_eval_predictions_gemma4_e2b_it_alexandria_eg_only_context3_all_group_r16_native_template_eos_4bit_lr5e5_resume_safe_v2_best_step1000.partial.csv
Total eval/test examples to evaluate: 1118
No existing prediction file found. Starting from scratch.
Remaining examples to generate: 1118


  0%|          | 0/1118 [00:00<?, ?it/s]

Saved partial predictions: 25 rows
Saved partial predictions: 50 rows
Saved partial predictions: 75 rows
Saved partial predictions: 100 rows
Saved partial predictions: 125 rows
Saved partial predictions: 150 rows
Saved partial predictions: 175 rows
Saved partial predictions: 200 rows
Saved partial predictions: 225 rows
Saved partial predictions: 250 rows
Saved partial predictions: 275 rows


### **Compute BLEU and chrF**

In [ ]:
# ============================================================
# Cell 20 — Compute BLEU / chrF / chrF++ on FULL eval predictions
# Strictly validates experiment, best_step, and full coverage
# ============================================================

import pandas as pd
import json
from pathlib import Path

try:
    import sacrebleu
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    import sacrebleu

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

if "BEST_CHECKPOINT_PATH" not in globals():
    raise RuntimeError("BEST_CHECKPOINT_PATH is not defined. Run Cell 17 first.")

if "BEST_EVAL_LOSS" not in globals():
    raise RuntimeError("BEST_EVAL_LOSS is not defined. Run Cell 17 first.")

eval_tag = f"best_step{BEST_STEP}"

pred_path = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{eval_tag}.csv"
metrics_path = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{eval_tag}.json"

print("Experiment:", EXPERIMENT_NAME)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
print("Best step:", BEST_STEP)
print("Prediction file:", pred_path)
print("Metrics file:", metrics_path)

if not pred_path.exists():
    raise FileNotFoundError(
        f"Prediction file not found: {pred_path}\n"
        "Run Cell 19 first."
    )

pred_df = pd.read_csv(pred_path)

required_cols = {
    "source_id",
    "prediction",
    "reference_arabic",
    "best_step",
    "model_checkpoint",
}

missing_cols = required_cols - set(pred_df.columns)

if missing_cols:
    raise ValueError(f"Missing required columns in prediction file: {missing_cols}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

before_dupes = len(pred_df)
pred_df = pred_df.drop_duplicates(subset=["source_id"], keep="first").copy()
after_dupes = len(pred_df)

if before_dupes != after_dupes:
    print(f"Removed duplicate prediction rows: {before_dupes - after_dupes}")

unique_steps = sorted(pred_df["best_step"].dropna().astype(str).unique().tolist())

if unique_steps != [str(BEST_STEP)]:
    raise RuntimeError(
        f"Prediction file has wrong or mixed best_step values: {unique_steps}. "
        f"Expected only: {BEST_STEP}."
    )

checkpoint_values = pred_df["model_checkpoint"].dropna().astype(str).unique().tolist()

if not any(f"checkpoint-{BEST_STEP}" in ckpt for ckpt in checkpoint_values):
    raise RuntimeError(
        "Prediction file does not appear to come from the current BEST checkpoint.\n"
        f"Expected checkpoint-{BEST_STEP} in model_checkpoint column.\n"
        f"Found values: {checkpoint_values[:5]}"
    )

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"].tolist())
actual_ids = set(pred_df["source_id"].tolist())

missing_ids = expected_ids - actual_ids
extra_ids = actual_ids - expected_ids

print("\n==============================")
print("Full Eval/Test Coverage Check")
print("==============================")
print("Expected eval/test examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Unique prediction source_ids:", len(actual_ids))

if missing_ids:
    raise RuntimeError(
        f"Prediction file is NOT full eval/test. "
        f"Missing {len(missing_ids)} examples. "
        f"Run Cell 19 again to finish generation."
    )

if extra_ids:
    print(f"Warning: prediction file has {len(extra_ids)} extra source_ids not in current eval_df.")

if "generation_error" in pred_df.columns:
    error_rows = pred_df["generation_error"].fillna("").astype(str).str.len() > 0
    if error_rows.any():
        print(f"Warning: {error_rows.sum()} rows contain generation_error.")

print("Full eval/test coverage confirmed.")

pred_df_ordered = expected_eval_df[["source_id"]].merge(
    pred_df,
    on = "source_id",
    how = "left",
)

preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

bleu = sacrebleu.corpus_bleu(preds, [refs])
chrf = sacrebleu.corpus_chrf(preds, [refs], word_order=0)
chrfpp = sacrebleu.corpus_chrf(preds, [refs], word_order=2)

metrics = {
    "experiment": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "checkpoint": str(BEST_CHECKPOINT_PATH),
    "best_step": int(BEST_STEP),
    "best_eval_loss": float(BEST_EVAL_LOSS),
    "num_examples": int(len(pred_df_ordered)),
    "unique_source_ids": int(pred_df_ordered["source_id"].nunique()),
    "BLEU": float(bleu.score),
    "chrF": float(chrf.score),
    "chrF++": float(chrfpp.score),
    "prediction_file": str(pred_path),
}

print("\n==============================")
print("Full Eval/Test Metrics from BEST checkpoint")
print("==============================")
print(f"Examples: {len(pred_df_ordered)}")
print(f"BLEU:     {bleu.score:.4f}")
print(f"chrF:     {chrf.score:.4f}")
print(f"chrF++:   {chrfpp.score:.4f}")

def compute_group_metrics(df, group_col):
    rows = []

    if group_col not in df.columns:
        return pd.DataFrame(rows)

    for group_value in sorted(df[group_col].dropna().unique()):
        tmp = df[df[group_col] == group_value]

        if len(tmp) == 0:
            continue

        group_preds = tmp["prediction"].fillna("").astype(str).tolist()
        group_refs = tmp["reference_arabic"].fillna("").astype(str).tolist()

        rows.append({
            group_col: group_value,
            "num_examples": int(len(tmp)),
            "BLEU": float(sacrebleu.corpus_bleu(group_preds, [group_refs]).score),
            "chrF": float(sacrebleu.corpus_chrf(group_preds, [group_refs], word_order=0).score),
            "chrF++": float(sacrebleu.corpus_chrf(group_preds, [group_refs], word_order=2).score),
        })

    return pd.DataFrame(rows)

per_config_df = compute_group_metrics(pred_df_ordered, "config")
per_dialect_df = compute_group_metrics(pred_df_ordered, "dialect")
per_domain_df = compute_group_metrics(pred_df_ordered, "domain")

if len(per_config_df):
    print("\n==============================")
    print("Per-config Metrics")
    print("==============================")
    display(per_config_df)
    metrics["per_config"] = per_config_df.to_dict("records")

if len(per_dialect_df):
    print("\n==============================")
    print("Per-dialect Metrics")
    print("==============================")
    display(per_dialect_df)
    metrics["per_dialect"] = per_dialect_df.to_dict("records")

if len(per_domain_df):
    print("\n==============================")
    print("Per-domain Metrics")
    print("==============================")
    display(per_domain_df)
    metrics["per_domain"] = per_domain_df.to_dict("records")

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("\nSaved metrics to:")
print(metrics_path)

display(pred_df_ordered[["source_text", "reference_arabic", "prediction"]].head(10))

### **Error-analysis view by country/domain**

In [ ]:
# ============================================================
# Cell 21 — Qualitative error-analysis samples
# Uses FULL eval predictions from BEST checkpoint
# ============================================================

if "BEST_STEP" not in globals():
    raise RuntimeError("BEST_STEP is not defined. Run Cell 17 first.")

eval_tag = f"best_step{BEST_STEP}"

analysis_path = PRED_DIR / f"manual_analysis_{EXPERIMENT_NAME}_{eval_tag}.xlsx"

if "pred_df_ordered" in globals():
    analysis_df = pred_df_ordered.copy()
elif "pred_df" in globals():
    analysis_df = pred_df.copy()
else:
    raise RuntimeError("No prediction dataframe found. Run Cell 19/20 first.")

with pd.ExcelWriter(analysis_path, engine="openpyxl") as writer:
    analysis_df.to_excel(writer, sheet_name="all_predictions", index=False)

    if "config" in analysis_df.columns:
        for cfg in analysis_df["config"].dropna().unique()[:10]:
            tmp = analysis_df[analysis_df["config"] == cfg]
            tmp.to_excel(writer, sheet_name=str(cfg)[:31], index=False)

print("Saved manual analysis workbook to:")
print(analysis_path)

### **Comparisons**